# Eval-set labeling UI

**For: Muhanad. Goal: hand-label 200–300 articles, blind, to ground-truth the LLM classifier.**

*Why blind?* The whole point of this eval set is to give us an *independent* ground truth — if we anchored on LLM suggestions, we'd be measuring agreement-with-itself, not classifier quality. Don't read anything beyond the snippet and URL before deciding. Don't second-guess to match what you think the model will say.

**How to use:**
1. Run all the cells below in order. The interactive widget appears in the last cell.
2. For each article: read the snippet + URL, tick the frame(s) you think are *foregrounded* (not just mentioned in passing). Multi-label is allowed — most articles have 1–2 frames.
3. Press **Save & Next** to record the labels and move on. Press **Skip** if the snippet is too thin to label — that's also a useful signal.
4. State is persisted after every save. You can close the notebook and resume any time; the widget reopens at the next unlabeled article.
5. If a snippet feels ambiguous, write a short note in the **Notes** box — boundary cases inform the taxonomy decision.

Codebook reminder lives in the cell directly above the widget, always visible while labeling.

See `data/external/codebook.md` for the frame definitions.

## Codebook (always visible)

| # | Frame | Definition (short) |
|---|-------|-------------------|
| 1 | **security** | Physical safety, public order, violence, intimidation, terrorism, armed conflict, policing, militia/insurgent activity. |
| 2 | **economy** | Macroeconomic conditions, fiscal/debt policy, currency, inflation, employment, growth, sector policy, business confidence. |
| 3 | **democracy** | Democratic norms, institutional health, constitutionalism, civil liberties, press freedom, backsliding/consolidation. *Distinct from process — democracy = whether the polity is democratic; process = whether this election was conducted cleanly.* |
| 4 | **identity** | Ethnicity, religion, region, language, generation, gender; identity-bloc politics, coalition arithmetic explained through identity. |
| 5 | **process** | Mechanics of this specific vote: registration, logistics, polling-day ops, counting, results transmission, court challenges, observer reports, alleged rigging. Procedural, not systemic. |
| 6 | **corruption** | Graft, kleptocracy, illicit enrichment, patronage networks, vote-buying, public-sector capture, accountability deficits framed as endemic. |

**Boundary-case heuristics** (full discussion in `data/external/codebook.md`):

- *security ↔ process*: election-day violence disrupting polling — pick **both** if the snippet foregrounds both physical safety and procedural integrity; pick the dominant one if clearly tilted.
- *democracy ↔ process*: a flawed count — `democracy` if the framing is systemic decline; `process` if it's about this election's specific count.
- *identity ↔ corruption*: ethnic-line patronage — assign both, note in the textbox.
- *economy ↔ corruption*: state-resource capture framed as mismanagement vs. as graft.

If you find yourself wanting to assign all 6, the snippet is too generic — press **Skip** and flag it in notes.

In [ ]:
"""Load candidate articles + initialize labeling state.

Candidates live in `data/external/eval_set_candidates.parquet` (stratified
sample produced by the cleaning pipeline). Labels are persisted to
`data/external/eval_set.parquet` — re-running the cell picks up at the next
unlabeled article.
"""
from pathlib import Path
import pandas as pd

EXTERNAL = Path("../data/external")
CANDIDATES_PATH = EXTERNAL / "eval_set_candidates.parquet"
LABELS_PATH = EXTERNAL / "eval_set.parquet"

FRAMES = ("security", "economy", "democracy", "identity", "process", "corruption")

candidates = pd.read_parquet(CANDIDATES_PATH)
print(f"Loaded {len(candidates):,} candidate articles.")

if LABELS_PATH.exists():
    existing = pd.read_parquet(LABELS_PATH)
    n_labeled = existing["labeled"].sum() if "labeled" in existing.columns else len(existing)
    print(f"Resuming: {n_labeled} of {len(candidates)} already labeled.")
else:
    existing = candidates.copy()
    existing["frame_labels"] = [[] for _ in range(len(existing))]
    existing["labeler_notes"] = ""
    existing["too_thin"] = False
    existing["labeled"] = False
    existing["labeled_at"] = pd.NaT
    existing.to_parquet(LABELS_PATH, index=False)
    print(f"Initialized fresh labeling file at {LABELS_PATH}.")


In [ ]:
"""Interactive labeling widget. Run this cell and the form appears below.

No LLM suggestions are loaded into the widget — labels are entered blind.
Every Save writes immediately to `eval_set.parquet`; you can close the
notebook any time.
"""
from datetime import datetime, timezone
from IPython.display import display, clear_output
import ipywidgets as widgets

state = {"df": pd.read_parquet(LABELS_PATH)}

def next_unlabeled_idx(df: pd.DataFrame) -> int | None:
    unlabeled = df.index[~df["labeled"]]
    return int(unlabeled[0]) if len(unlabeled) else None

progress = widgets.HTML()
election_html = widgets.HTML()
url_html = widgets.HTML()
snippet_html = widgets.HTML()
checkboxes = {f: widgets.Checkbox(value=False, description=f, indent=False) for f in FRAMES}
too_thin = widgets.Checkbox(value=False, description="Too thin / off-topic — skip", indent=False)
notes = widgets.Textarea(placeholder="Boundary-case observations, ambiguity notes...",
                          layout=widgets.Layout(width="100%", height="60px"))
save_btn = widgets.Button(description="Save & Next", button_style="primary")
skip_btn = widgets.Button(description="Skip (no label)", button_style="warning")
msg = widgets.HTML()

def render(idx: int) -> None:
    df = state["df"]
    n_done = int(df["labeled"].sum())
    progress.value = f"<b>Progress:</b> {n_done} of {len(df)} labeled ({n_done/len(df)*100:.0f}%)"
    if idx is None:
        election_html.value = "<h3>All articles labeled. Thank you!</h3>"
        url_html.value = snippet_html.value = ""
        for cb in checkboxes.values(): cb.disabled = True
        too_thin.disabled = notes.disabled = save_btn.disabled = skip_btn.disabled = True
        return
    row = df.loc[idx]
    election_html.value = (f"<h3>Article {idx+1}/{len(df)} — {row['election']} "
                            f"({row['outlet_origin']}, {row['SourceCommonName']})</h3>")
    url_html.value = f"<a href='{row['DocumentIdentifier']}' target='_blank'>{row['DocumentIdentifier']}</a>"
    snippet_html.value = (f"<div style='background:#f6f6f6;padding:10px;border-radius:4px;"
                           f"white-space:pre-wrap;font-family:sans-serif'>{row['text_snippet']}</div>")
    for cb in checkboxes.values(): cb.value = False
    too_thin.value = False
    notes.value = ""
    msg.value = ""

def save_current(_btn) -> None:
    df = state["df"]
    idx = next_unlabeled_idx(df)
    if idx is None: return
    chosen = [f for f, cb in checkboxes.items() if cb.value]
    if not chosen and not too_thin.value:
        msg.value = "<span style='color:#b00'>Pick at least one frame, or tick 'Too thin'.</span>"
        return
    df.at[idx, "frame_labels"] = chosen
    df.at[idx, "too_thin"] = bool(too_thin.value)
    df.at[idx, "labeler_notes"] = notes.value
    df.at[idx, "labeled"] = True
    df.at[idx, "labeled_at"] = datetime.now(timezone.utc)
    df.to_parquet(LABELS_PATH, index=False)
    state["df"] = df
    render(next_unlabeled_idx(df))

def skip_current(_btn) -> None:
    df = state["df"]
    idx = next_unlabeled_idx(df)
    if idx is None: return
    df.at[idx, "frame_labels"] = []
    df.at[idx, "too_thin"] = True
    df.at[idx, "labeler_notes"] = notes.value
    df.at[idx, "labeled"] = True
    df.at[idx, "labeled_at"] = datetime.now(timezone.utc)
    df.to_parquet(LABELS_PATH, index=False)
    state["df"] = df
    render(next_unlabeled_idx(df))

save_btn.on_click(save_current)
skip_btn.on_click(skip_current)

frame_box = widgets.VBox([
    widgets.HTML("<b>Which frame(s) are foregrounded?</b> (multi-label allowed)"),
    widgets.HBox(list(checkboxes.values())),
    too_thin,
])
form = widgets.VBox([
    progress, election_html, url_html, snippet_html,
    frame_box,
    widgets.HTML("<b>Notes (optional):</b>"), notes,
    widgets.HBox([save_btn, skip_btn]), msg,
])
render(next_unlabeled_idx(state["df"]))
display(form)
